# 🏦 Investment Research with Compliance Plan Review

## Overview

This notebook demonstrates **interactive plan review** in Magentic workflows for financial services - enabling compliance oversight and modification of research plans before execution. This pattern is essential for:

1. **Regulatory Compliance**: Ensure research follows SEC/FINRA guidelines
2. **Risk Management**: Human approval for client-facing materials
3. **Quality Assurance**: Verify AI-generated research methodology
4. **Iterative Refinement**: Collaborate with AI to optimize research approach

> ⚠️ **Financial Services Disclaimer**: This notebook demonstrates AI agent workflows for educational purposes. In production, all investment research must be reviewed by compliance before distribution.

### Industry Use Case: Compliance-Reviewed Investment Research

A wealth management firm requires compliance review of AI-generated research plans before execution to ensure:
- Research methodology meets regulatory standards
- Appropriate disclosures are included
- Data sources are approved and documented
- Client suitability considerations are addressed

### Key Concepts

| Concept | Description |
|---------|-------------|
| **`.with_plan_review()`** | Enables human review of generated plans |
| **`MagenticPlanReviewRequest`** | Event type for plan review requests |
| **`event_data.approve()`** | Accept the proposed plan |
| **`event_data.revise(feedback)`** | Provide feedback to modify the plan |

### Plan Review Lifecycle

```
Investment Research Request
           ↓
Orchestrator Generates Research Plan
           ↓
Request Compliance Review (with_plan_review)
           ↓
┌─────────────────────────────────┐
│   Compliance Reviews Plan       │
│  ├── approve() - Accept plan    │
│  └── revise(feedback) - Modify  │
└─────────────────────────────────┘
           ↓
Resume with Compliance Decision
           ↓
Execute Plan → Final Research Report
```

### Review Response Options

| Method | Action | Industry Use Case |
|--------|--------|--------------|
| **`approve()`** | Accept plan, continue execution | Research plan meets compliance standards |
| **`revise(feedback)`** | Modify plan with feedback | Add disclosures or adjust methodology |

## Prerequisites

- ✅ Azure AI Foundry configured
- ✅ Environment variables: `AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- ✅ Azure CLI authentication: Run `az login`

## 1️⃣ Setup and Imports

In [1]:
import asyncio
import json
import logging
from typing import cast
from dataclasses import dataclass

import os
from dotenv import load_dotenv
from agent_framework import (
    Agent,
    AgentResponseUpdate,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,
    WorkflowRunState,
    handler,
    response_handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

# Load environment variables from .env file
load_dotenv('../../.env')

True

## 2️⃣ Create Specialized Financial Agents

Multi-agent setup for investment research:
- **MarketResearcherAgent**: Market data, sector analysis, news gathering
- **QuantAnalystAgent**: Data analysis, summarization, insights
- **MagenticManager**: Orchestrator that coordinates the workflow

In [2]:
# Create Foundry chat client
PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
chat_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT,
    credential=AzureCliCredential()
)
print("✅ Foundry Chat Client created")

# Create specialized financial agents
market_researcher = Agent(
    client=chat_client,
    name="MarketResearcherAgent",
    description="Specialist in market research, sector analysis, and financial news gathering",
    instructions=(
        "You are a Financial Market Researcher at a wealth management firm. "
        "You gather market data, sector trends, company news, and economic indicators. "
        "Focus on factual information without performing calculations. "
        "Always cite sources and note the date of any market data you reference."
    ),
)
print("✅ Market Researcher Agent created")

quant_analyst = Agent(
    client=chat_client,
    name="QuantAnalystAgent",
    description="Data analyst who processes and summarizes research findings",
    instructions=(
        "You are a Quantitative Analyst at a wealth management firm. "
        "You analyze findings and create summaries with risk metrics. "
        "Always show your calculation methodology and assumptions."
    ),
)
print("✅ Quant Analyst Agent created")

✅ Foundry Chat Client created
✅ Market Researcher Agent created
✅ Quant Analyst Agent created


## 3️⃣ Build Workflow with Plan Review

Use `.with_plan_review()` to enable compliance review of generated plans before execution.

In [3]:
print("\n🏦 Building Investment Research Workflow with Compliance Review...")

@dataclass
class PlanReviewRequest:
    """Request for compliance review of the research plan."""
    plan: str
    
@dataclass
class PlanReviewResponse:
    """Compliance officer's response to the plan review."""
    approved: bool
    feedback: str = ""


class OrchestratorExecutor(Executor):
    """Orchestrator that coordinates research agents and requests compliance plan review."""
    
    def __init__(self, id: str, chat_client, market_researcher: Agent, quant_analyst: Agent):
        super().__init__(id=id)
        self._chat_client = chat_client
        self._market_researcher = market_researcher
        self._quant_analyst = quant_analyst
    
    @handler
    async def start(self, research_request: str, ctx: WorkflowContext) -> None:
        """Generate a research plan and request compliance review."""
        # Generate plan using the orchestrator
        plan_messages = [
            Message(role="system", text=(
                "You coordinate a team of financial analysts. "
                "Create a research plan with numbered steps for the team to follow. "
                "Available agents: MarketResearcher (data/trends), QuantAnalyst (analysis/metrics). "
                "Ensure all research includes appropriate risk disclosures."
            )),
            Message(role="user", text=research_request),
        ]
        plan_response = await self._chat_client.get_response(messages=plan_messages)
        plan_text = plan_response.messages[-1].text or ""
        
        print(f"\n📋 Generated Research Plan")
        
        # Request compliance review
        await ctx.request_info(
            request_data=PlanReviewRequest(plan=plan_text),
            response_type=PlanReviewResponse,
        )
    
    @response_handler
    async def on_compliance_decision(
        self,
        original_request: PlanReviewRequest,
        response: PlanReviewResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Handle compliance decision and execute research if approved."""
        if not response.approved:
            # Revise the plan
            revised_messages = [
                Message(role="system", text=(
                    "You coordinate financial analysts. Revise the research plan based on compliance feedback."
                )),
                Message(role="user", text=f"Original plan:\n{original_request.plan}\n\nCompliance feedback: {response.feedback}\n\nPlease revise the plan."),
            ]
            revised_response = await self._chat_client.get_response(messages=revised_messages)
            revised_plan = revised_response.messages[-1].text or ""
            
            print(f"\n📋 Revised Research Plan")
            
            # Request review again
            await ctx.request_info(
                request_data=PlanReviewRequest(plan=revised_plan),
                response_type=PlanReviewResponse,
            )
            return
        
        # Plan approved - execute research
        print("\n✅ Plan approved - executing research...")
        
        # Step 1: Market Research
        print("\n📊 MarketResearcherAgent working...")
        research_response = await self._market_researcher.run(
            f"Based on this approved plan, conduct market research:\n\n{original_request.plan}"
        )
        market_findings = research_response.text or ""
        
        # Step 2: Quantitative Analysis
        print("\n📊 QuantAnalystAgent working...")
        analysis_response = await self._quant_analyst.run(
            f"Based on these market research findings, provide quantitative analysis and summary:\n\n{market_findings}"
        )
        final_report = analysis_response.text or ""
        
        await ctx.yield_output(final_report)


orchestrator = OrchestratorExecutor(
    id="orchestrator", 
    chat_client=chat_client, 
    market_researcher=market_researcher, 
    quant_analyst=quant_analyst,
)

workflow = WorkflowBuilder(start_executor=orchestrator).build()

print("✅ Workflow built with compliance plan review enabled")


🏦 Building Investment Research Workflow with Compliance Review...
✅ Workflow built with compliance plan review enabled


## 4️⃣ Define Investment Research Request

In [4]:
# Client research request from a wealth advisor
research_request = (
    "Research the current state of sustainable investment funds (ESG) "
    "and summarize the findings for a client considering adding ESG exposure "
    "to their portfolio. Include recent performance trends and risk considerations."
)

print(f"📋 Research Request: {research_request}")

📋 Research Request: Research the current state of sustainable investment funds (ESG) and summarize the findings for a client considering adding ESG exposure to their portfolio. Include recent performance trends and risk considerations.


## 5️⃣ Run the Compliance-Reviewed Research Workflow

### ⚠️ IMPORTANT: How to Respond to Plan Review

When prompted for plan review, **an input box will appear at the top of VS Code**.

> **You MUST type your response in the input box, then press Enter.**  
> - Press **Enter with empty input** to **approve** the plan
> - Type **feedback text** then Enter to **revise** the plan with your feedback

### Example Responses:

| Input | Action |
|-------|--------|
| *(empty - just press Enter)* | Approve the plan as-is |
| `Add risk disclosures` | Revise plan with your feedback |
| `Include diversification recommendations` | Revise plan with your feedback |

In [5]:
async def run_compliance_reviewed_research():
    """Run the investment research workflow with compliance plan review."""
    
    print("\n🚀 Starting research workflow (compliance review required)...")
    print("=" * 60)
    
    pending_responses: dict[str, object] | None = None
    output_data = None
    review_count = 0
    
    while output_data is None:
        # First iteration uses run with research request
        # Subsequent iterations use run with responses
        if pending_responses is not None:
            stream = workflow.run(responses=pending_responses, stream=True)
        else:
            stream = workflow.run(research_request, stream=True)
        
        requests: list[tuple[str, PlanReviewRequest]] = []
        async for event in stream:
            if event.type == "data" and isinstance(event.data, AgentResponseUpdate):
                if event.data.text:
                    print(event.data.text, end="", flush=True)
            
            elif event.type == "request_info" and isinstance(event.data, PlanReviewRequest):
                requests.append((event.request_id, event.data))
            
            elif event.type == "output":
                output_data = event.data
        
        pending_responses = None
        
        # Handle plan review request if any
        if requests and output_data is None:
            for request_id, plan_request in requests:
                review_count += 1
                
                print("\n\n" + "=" * 60)
                print(f"🏦 COMPLIANCE PLAN REVIEW #{review_count}")
                print("=" * 60)
                
                print(f"\n📋 Proposed Plan:\n{plan_request.plan}\n")
                print("-" * 60)
                print("👨‍💼 COMPLIANCE OFFICER: Review the plan above")
                print("-" * 60)
                
                reply = await asyncio.get_event_loop().run_in_executor(
                    None, 
                    input, 
                    "Enter APPROVE (or empty) to approve, or type feedback to revise: "
                )
                
                if reply.strip() == "" or reply.strip().upper() == "APPROVE":
                    print("\n✅ Plan APPROVED by compliance.\n")
                    pending_responses = {request_id: PlanReviewResponse(approved=True)}
                else:
                    print(f"\n📝 Plan REVISED with feedback: {reply}\n")
                    pending_responses = {request_id: PlanReviewResponse(approved=False, feedback=reply)}
    
    # Display final output
    print("\n" + "=" * 60)
    print("📊 INVESTMENT RESEARCH REPORT COMPLETE")
    print("=" * 60)
    print(output_data)
    
    print(f"\n✅ Completed with {review_count} compliance review(s)")
    print("\n" + "=" * 60)
    print("⚠️ DISCLAIMER: This is for demonstration purposes only.")
    print("   Actual investment research requires full compliance review.")

## 6️⃣ Run the Research Workflow

In [6]:
await run_compliance_reviewed_research()


🚀 Starting research workflow (compliance review required)...

📋 Generated Research Plan


🏦 COMPLIANCE PLAN REVIEW #1

📋 Proposed Plan:
### Research Plan: Sustainable Investment Funds (ESG)

**Objective:** 
Provide a comprehensive report on the current state of sustainable investment funds (ESG), including recent performance trends and associated risks, enabling the client to make an informed decision about adding ESG exposure to their portfolio.

---

### Available Team:
- **MarketResearcher:** Responsible for gathering data, industry trends, and market dynamics related to ESG funds.
- **QuantAnalyst:** Responsible for analyzing metrics, interpreting performance data, and assessing quantitative risks.

---

### Research Plan Steps: 

#### Step 1: Define Scope and Objectives
- **Responsible:** Team Lead
  1. Outline the client's needs and objectives for ESG exposure.
  2. Define a clear time frame for research (e.g., 12 months, 3 years, 5 years of performance data).
  3. Identify targ

## 📝 Key Takeaways

### Compliance Plan Review for FSI

| Benefit | Description |
|---------|-------------|
| **Regulatory Compliance** | Ensure research meets SEC/FINRA guidelines |
| **Quality Assurance** | Verify methodology before execution |
| **Risk Management** | Human approval for client-facing materials |
| **Audit Trail** | Document compliance decisions |

### Industry Use Cases for Plan Review

| Use Case | Review Focus |
|----------|-------------|
| Investment Research | Methodology, disclosures, suitability |
| Client Communications | Regulatory compliance, accuracy |
| Trade Recommendations | Risk assessment, conflict checks |
| Financial Reporting | Data sources, calculation methods |

### Production Considerations

| Consideration | Recommendation |
|---------------|----------------|
| **Audit Trail** | Log all compliance decisions with timestamps |
| **Timeout Handling** | Auto-escalate if review not completed in SLA |
| **Role-Based Access** | Only authorized compliance officers can approve |
| **Multi-Level Approval** | Senior approval for high-value research |